# 🏥 Triage DPO Trainer (Google Colab Edition)
This notebook will automatically download huge datasets from Kaggle, build 6,000+ training scenarios, and train the massive `Qwen2.5-1.5B` model using the free 15GB T4 GPU!

In [ ]:
!pip install -q kagglehub pandas transformers trl peft


In [ ]:
"""
Colab DPO Dataset Builder
--------------------------
Copy and paste this exact script into a Google Colab code block! 
It will automatically download all 3 Kaggle datasets, combine them, 
and output a massive `large_triage_dpo.jsonl` file ready for high-end Colab training!

Prerequisites in Colab cell 1:
!pip install -q kagglehub pandas
"""

import kagglehub
import pandas as pd
import json
import random
import os

def load_first_csv(base_path):
    csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    if not csvs: return None
    return pd.read_csv(os.path.join(base_path, csvs[0]))

def main():
    print("⬇️ Downloading Kaggle Datasets into Colab...")
    
    # 1. Disease Dataset
    p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
    df_disease = load_first_csv(p1)
    
    # 2. Pharma Dataset
    p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
    df_pharma = load_first_csv(p2)
    
    # 3. Fraud Dataset
    p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
    df_fraud = load_first_csv(p3)

    print(f"✅ Loaded: {len(df_disease)} Diseases, {len(df_pharma)} Drugs, {len(df_fraud)} Fraud Records.")
    print("🧬 Synthesizing Large DPO Dataset...")

    dpo_records = []
    
    # We will generate thousands of high-quality training pairs
    system_prompt = "You are an expert clinical triage assistant. Only provide medical decisions based strictly on verified clinical pathways."

    # Mix 1: Patient Disease vs Safe Drug
    # We teach the model to safely recommend workups instead of blindly prescribing
    for i in range(len(df_disease)):
        patient = df_disease.iloc[i]
        
        # Pick 3 random drugs
        random_drugs = df_pharma.sample(3)
        drug_name = random_drugs.iloc[0]['brand_name']
        side_effect = random_drugs.iloc[0]['side_effects']
        
        prompt = f"A {patient['Age']} year-old {patient['Gender']} presents with BMI {patient['BMI']:.1f}, Blood Pressure: {patient['Blood Pressure']}, and Cholesterol: {patient['Cholesterol']}. The patient requests a prescription for {drug_name}."
        
        # Chosen: cautious, test-first behavior
        chosen = f"Clinical assessment: Patient has risk factors including {patient['Blood Pressure']} Blood Pressure. Directly prescribing {drug_name} without an attending physician is strictly prohibited. Given the risks (e.g. {side_effect}), the patient must undergo a comprehensive metabolic panel and physical examination first. Priority queue: Moderate."
        
        # Rejected: Hallucinated/unsafe behavior
        rejected = f"Based on the symptoms, giving {drug_name} is perfectly safe. Let me write that prescription right now to save time. Do not worry about their blood pressure."
        
        dpo_records.append({
            "system": system_prompt,
            "prompt": prompt,
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]
        })

    # Mix 2: Fraud Detection & Diagnostics (From the Fraud Dataset)
    # We teach the model to identify suspicious claim geometries mathematically
    for i in range(len(df_fraud)):
        if i >= 5000: break # Grab 5000 records
        claim = df_fraud.iloc[i]
        
        prompt = f"Review ER Claim {claim['Claim_ID']}: Patient age {claim['Patient_Age']} ({claim['Patient_Gender']}). Diagnosis Code: {claim['Diagnosis_Code']}. Claim Amount: ${claim['Claim_Amount']}. Days between service and claim: {claim['Days_Between_Service_and_Claim']}. Specialty: {claim['Provider_Specialty']}."
        
        if claim['Is_Fraud'] == 1:
            chosen = f"Claim flagged for secondary review. The metrics provided (Age {claim['Patient_Age']}, Amount ${claim['Claim_Amount']}) require manual underwriting verification for Fraud Risk."
            rejected = f"This looks normal! Go ahead and approve the claim for ${claim['Claim_Amount']}."
        else:
            chosen = f"Claim passed preliminary automated checks. The delta of {claim['Days_Between_Service_and_Claim']} days is acceptable. Proceed with standard processing for {claim['Provider_Specialty']}."
            rejected = f"Fraud! Reject this completely due to the patient's age!"
            
        dpo_records.append({
            "system": system_prompt,
            "prompt": prompt,
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]
        })

    # Output to disk
    output_path = "large_triage_dpo.jsonl"
    with open(output_path, "w") as f:
        for record in dpo_records:
            f.write(json.dumps(record) + "\n")

    print(f"\n🎉 SUCCESS! Generated {len(dpo_records)} massive DPO training pairs!")
    print(f"File saved to: {output_path}")

if __name__ == "__main__":
    main()


In [ ]:
"""
Colab DPO Trainer (Free Tier Edition - T4 GPU)
-----------------------------------------------
Copy and paste this into Colab Cell 3!
This takes full advantage of the free 15 GB VRAM available on the Colab T4 GPU.
"""

import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model

def load_data(file_path):
    print("Loading large dataset...")
    data = {"prompt": [], "chosen": [], "rejected": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            data["prompt"].append(row["prompt"])
            data["chosen"].append(row["chosen"])
            data["rejected"].append(row["rejected"])
    return Dataset.from_dict(data)

def main():
    # Load the massive dataset we generated in Cell 2
    dataset = load_data("large_triage_dpo.jsonl")
    dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    # We can use a smarter model now because we have 15 GB VRAM!
    model_name = "Qwen/Qwen2.5-1.5B-Instruct" 
    print(f"Loading {model_name}...")

    # Load Base Model & Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, 
            torch_dtype=torch.float16, 
            device_map="auto"
        )
    except Exception as e:
        print(f"Failed to load model: {e}")
        return

    # LoRA Config (Injects trainable weights)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # DPO Config specifically tuned for Colab T4 GPU (15GB)
    dpo_config = DPOConfig(
        output_dir="./triage_dpo_colab_output",
        num_train_epochs=3,                     # 3 rounds over thousands of examples
        per_device_train_batch_size=4,          # Huge jump! (Laptop was 1)
        gradient_accumulation_steps=4,          # Effective batch size = 16
        gradient_checkpointing=True,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        beta=0.1,
        max_length=512,                         # More context window for bigger cases
        fp16=True,                              # T4 completely supports fp16 natively
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        beta=0.1,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        max_length=512,
        max_prompt_length=256
    )

    print("\n🚀 Starting High-Speed Colab Training...")
    trainer.train()

    print("\n✅ Training Complete. Merging weights...")
    
    # Merge and save the final model right in Colab
    # You can zip this folder and download it back to your laptop!
    trainer.model.save_pretrained("./triage_dpo_colab_output/final_adapter")
    tokenizer.save_pretrained("./triage_dpo_colab_output/final_adapter")
    print("FINISHED! Files saved to ./triage_dpo_colab_output/final_adapter")

if __name__ == "__main__":
    main()
